In [1]:
import os
os.environ.pop('SIRL_USE_JAX', None)
#os.environ['SIRL_USE_JAX'] = "0"

import articulated_dynamics.visualizer as viz
from articulated_dynamics.robots import CartPole, ThreeLinks
from articulated_dynamics.articulated_system import State
from articulated_dynamics.kinematics import Kinematics
from articulated_dynamics.dynamics import Dynamics
from articulated_dynamics.integrator import integrate_euler, integrate_implicit
from articulated_dynamics.math_utils import nplib
import numpy as np

import mujoco



In [2]:
sys = ThreeLinks()

three_link_xml = """
<mujoco>
  <option gravity="0 -9.81 0" />
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="512" />
    <texture name="plane" type="2d" builtin="checker" rgb1=".2 .3 .4" rgb2=".1 0.15 0.2" width="512" height="512" mark="cross" markrgb=".8 .8 .8" />
    <material name="plane" reflectance="0.3" texture="plane" texrepeat="1 1" texuniform="true" />
  </asset>

  <worldbody>
    <light directional="true" diffuse=".2 .2 .2" specular="0 0 0" pos="0 1 5" dir="0 -1 -1" castshadow="false"/>
    <light directional="false" diffuse=".8 .8 .8" specular="0.3 0.3 0.3" pos="0 -1 4" dir="0 0 -1" />
    <light directional="true" diffuse="0 0 0" specular=".7 .7 .7" pos="0 3 3" dir="0 -3 -3" />

    <body name="link_0" pos="0 0.15 0" euler="0 0 0">
        <joint name="joint_0" type="hinge" axis="0 0 1" pos="0 -0.15 0"/>
        <geom name="box_link_0" type="box" size=".025 .15 .025" rgba="0.8 0.8 0.8 1" density="2700"/>

        <body name="link_1" pos="0 0.3 0">
              <joint name="joint_1" type="hinge" axis="0 0 1" pos="0 -0.15 0"/>
              <geom name="box_link_1" type="box" size=".025 .15 .025" rgba="0.8 0.8 0.8 1" density="2700"/>

              <body name="link_2" pos="0 0.3 0">
                <joint name="joint_2" type="hinge" axis="0 0 1" pos="0 -0.15 0"/>
                <geom name="box_link_2" type="box" size=".025 .15 .025" rgba="0.8 0.8 0.8 1" density="2700"/>
              </body>
        </body>
    </body>
  </worldbody>
</mujoco>
"""






In [3]:
viewer = viz.P3JSViewer(width=300, height=400)
viewer.create_shapes(sys)
q = nplib.array([-3.1415926/6, 3.1415926/3, -3.1415926/6])
qd = nplib.array([-0.8, 0.5, 0.5])
qdd = nplib.array([-0.2, -0.3, 0.1])
x, xd, xdd = Kinematics.forward(sys, q, qd, qdd)
tau = Dynamics.inverse(sys, q, qd, qdd)
M = Dynamics.mass(sys, q)

s = State(q=q, qd=qd, x=x, xd=xd)
viewer.place_shapes(sys, s)
viewer.show()

link_id = -1
print(x[link_id])
print(xd[link_id])
print(xdd[link_id])
print(tau)
print(M)
print("fwd qdd:", Dynamics.forward(sys, q, qd, tau))

#check results from mujoco
mjc_model = mujoco.MjModel.from_xml_string(three_link_xml)
mjc_data = mujoco.MjData(mjc_model)

mjc_data.qpos = list(nplib.asarray(q))
mjc_data.qvel = list(nplib.asarray(qd))
mjc_data.qacc = list(nplib.asarray(qdd))

#mujoco.mj_forward(mjc_model, mjc_data)
mujoco_tau = np.zeros(3)
mujoco.mj_inverse(mjc_model, mjc_data)
#need to call this for mj_objectAcceleration to work, see mujoco API doc
mujoco.mj_rnePostConstraint(mjc_model, mjc_data)

mujoco_link_vel = np.zeros(6)
mujoco_link_acc = np.zeros(6)
#print("nbr of mjc bodies:", mjc_model.nbody)
print("----------results from mujoco------------")
mujoco.mj_objectVelocity(mjc_model, mjc_data, mujoco.mjtObj.mjOBJ_BODY, link_id % mjc_model.nbody, mujoco_link_vel, False)
mujoco.mj_objectAcceleration(mjc_model, mjc_data, mujoco.mjtObj.mjOBJ_BODY, link_id % mjc_model.nbody, mujoco_link_acc, False)

print(list(mjc_data.xpos[link_id]) +  list(mjc_data.xquat[link_id]))
print(mujoco_link_vel)
print(mujoco_link_acc)
print(mjc_data.qfrc_inverse)

mujoco_M = np.zeros((3,3))
mujoco.mj_fullM(mjc_model, mujoco_M, mjc_data.qM)
#print(mujoco_M[0])
print(mujoco_M)

print("---------jacobian test-----------------")
#try a specific col with forward kinematics at velocity level
q = nplib.array([-3.1415926/6, 3.1415926/3, -3.1415926/6])
qd = nplib.array([0, 0, 1])
qdd = nplib.array([0, 0, 0])

x_ws_lst, x_ps_lst, xd_ss_lst, xdd_ss_lst, _ = Kinematics._forward(sys, q, qd, qdd)
print(xd_ss_lst[2])
jac = Kinematics.jacobian(sys, q, body_com=True)
print(jac[2])

Renderer(camera=PerspectiveCamera(aspect=0.75, children=(DirectionalLight(color='white', intensity=0.6, positi…

Transform(trans=array([9.71445147e-17, 6.69615245e-01, 0.00000000e+00]), rot=array([ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00, -8.32667268e-17]))
Motion(ang=array([0. , 0. , 0.2]), lin=array([ 0.25578838, -0.075     ,  0.        ]))
Motion(ang=array([ 0. ,  0. , -0.4]), lin=array([0.15936534, 9.65934043, 0.        ]))
[ 2.689689   -4.55160491 -0.05465097]
[[1.31930689 0.67803095 0.21900501]
 [0.67803095 0.46242688 0.14008844]
 [0.21900501 0.14008844 0.06117187]]
fwd qdd: [-0.2 -0.3  0.1]
----------results from mujoco------------
[np.float64(7.130394446416029e-17), np.float64(0.6696152449501528), np.float64(0.0), np.float64(0.9999999999999999), np.float64(0.0), np.float64(0.0), np.float64(-5.2642644109674864e-17)]
[ 0.          0.          0.2         0.25578838 -0.075       0.        ]
[ 0.          0.         -0.4         0.15936534  9.65934043  0.        ]
[ 2.689689   -4.55160491 -0.05465097]
[[1.31930689 0.67803095 0.21900501]
 [0.67803095 0.46242688 0.14008844]
 [0.2190050

In [4]:
import time

T = 100
dt = 0.01
q0 = nplib.array([5*3.1415926/6, 3.1415926/3, -3.1415926/6])
qd0 = nplib.zeros(3)
tau = nplib.zeros(3)

q_traj = [q0]
qd_traj = [qd0]

state_traj = []
for t in range(T):
    qdd = Dynamics.forward(sys, q_traj[-1], qd_traj[-1], tau)
    q_next, qd_next = integrate_euler(sys, dt, q_traj[-1], qd_traj[-1], qdd, symplectic=False)
    q_traj.append(q_next)
    qd_traj.append(qd_next)

    
    # viewer.place_shapes_qpos(sys, q_next)
    # if t == 0:
    #     viewer.show()
    # time.sleep(0.01)




In [5]:
anim_action = viewer.animate_qpos_traj(sys, q_traj, dt)
viewer.show()
anim_action

Renderer(camera=PerspectiveCamera(aspect=0.75, children=(DirectionalLight(color='white', intensity=0.6, matrix…

AnimationAction(clip=AnimationClip(duration=1.01, tracks=(VectorKeyframeTrack(name='scene/link_0.position', ti…

In [6]:
sys = CartPole()
viewer = viz.P3JSViewer(width=600, height=480)
viewer.create_shapes(sys)

q0 = nplib.array([0, 3.1415926/3])
qd0 = nplib.zeros(2)

T = 50
dt = 0.1

tau = nplib.zeros(2)

q_traj = [q0]
qd_traj = [qd0]

state_traj = []
for t in range(T):
    #qdd = Dynamics.forward(sys, q_traj[-1], qd_traj[-1], tau)
    #q_next, qd_next = integrate_euler(sys, dt, q_traj[-1], qd_traj[-1], qdd, symplectic=False)
    q_next, qd_next = integrate_implicit(sys, dt, q_traj[-1], qd_traj[-1], tau)
    q_traj.append(q_next)
    qd_traj.append(qd_next)

viewer.show()
anim_action = viewer.animate_qpos_traj(sys, q_traj, dt)
anim_action


/Users/hang/miniconda3/envs/sirl/lib/python3.10/site-packages/scipy-1.14.0-py3.10-macosx-11.0-arm64.egg/scipy/optimize/_nonlin.py:374: RuntimeWarning: invalid value encountered in scalar divide
  and dx_norm/self.x_rtol <= x_norm))


Renderer(camera=PerspectiveCamera(aspect=1.25, children=(DirectionalLight(color='white', intensity=0.6, positi…

AnimationAction(clip=AnimationClip(duration=5.1000000000000005, tracks=(VectorKeyframeTrack(name='scene/cart.p…

In [7]:
sys = ThreeLinks()
viewer = viz.P3JSViewer(width=600, height=480)
viewer.create_shapes(sys)

q0 = nplib.array([-nplib.pi/2, 0, 0])
qd0 = nplib.zeros(3)

T = 50
dt = 0.1

tau = nplib.zeros(3)

q_traj = [q0]
qd_traj = [qd0]

state_traj = []
for t in range(T):
    #qdd = Dynamics.forward(sys, q_traj[-1], qd_traj[-1], tau)
    #q_next, qd_next = integrate_euler(sys, dt, q_traj[-1], qd_traj[-1], qdd, symplectic=False)
    q_next, qd_next = integrate_implicit(sys, dt, q_traj[-1], qd_traj[-1], tau)
    q_traj.append(q_next)
    qd_traj.append(qd_next)

viewer.show()
anim_action = viewer.animate_qpos_traj(sys, q_traj, dt)
anim_action

Renderer(camera=PerspectiveCamera(aspect=1.25, children=(DirectionalLight(color='white', intensity=0.6, positi…

AnimationAction(clip=AnimationClip(duration=5.1000000000000005, tracks=(VectorKeyframeTrack(name='scene/link_0…

In [2]:
#test new spatial inertia multiplicatio mapping impl
import articulated_dynamics.math_utils as math
import articulated_dynamics.spatial_algebra as sa
import numpy as np

ang = (np.random.rand(3) - 0.5) * 5
lin = (np.random.rand(3) - 0.5) * 5
inertia = sa.Inertia.from_it_and_mass(it=nplib.eye(3)/6., mass=2.7)
motion = sa.Motion(ang=ang, lin=lin)

print(inertia.mul(motion))
axis = np.random.rand(3)
axis = axis / np.linalg.norm(axis)
transform = sa.Transform(rot=math.quat_rot_axis(axis, np.random.rand()), trans=np.random.rand(3))
m1 = motion.apply_transform(transform)
m2 = transform.create_inverse().to_6dmat()@motion.to_6darray()
print(m1, m2)

t = sa.Transform(trans=nplib.array([1, 0, 0]))
print(t.to_6dmat())

Force(ang=array([0.09211176, 0.29953675, 0.21162149]), lin=array([-5.58224116, -2.56572162, -4.00499368]))
Motion(ang=array([0.63888221, 1.65422297, 1.41528575]), lin=array([-0.92528112, -0.53999625, -2.29599767])) [ 0.63888221  1.65422297  1.41528575 -2.20500816 -0.56870707 -1.69833695]
[[ 1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.]
 [ 0.  0.  1.  0.  1.  0.]
 [ 0. -1.  0.  0.  0.  1.]]


In [6]:
rot_mat = math.rotate(nplib.eye(3), transform.rot)
rot = math.quat_from_mat(rot_mat.T)
print(rot_mat)
print(transform.rot, rot)

[[ 0.84501139  0.36904065 -0.3869945 ]
 [-0.34314716  0.92925528  0.1368746 ]
 [ 0.41012898  0.01713547  0.91186655]]
[0.95996526 0.03118319 0.20759175 0.18547229] [0.95996526 0.03118319 0.20759175 0.18547229]
